<a href="https://colab.research.google.com/github/jihanbakshi7/Explainable-Multi-modal-Fact-Verification-System-Using-Context-Aware-Retrieval-Augmented-Evi.-Gen./blob/main/notebooks/03_baseline_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NOTEBOOK 3 / 6 — Baseline Pipeline (fresh build)
**Runtime: GPU (T4) REQUIRED.** Heaviest notebook so far -- one 4-bit LLM
loaded, decomposition + verdict generation per claim.

**Requires:** Notebooks 1 and 2's outputs in the shared Drive checkpoint
folder.

Reproduces the baseline: decompose -> reflect -> FIRE confidence gate ->
flat-evidence atomic verdict -> deterministic 4-class aggregation. This is
the number your later novelty notebooks get compared against.

## Carried forward from Notebook 2's fixes
- Every model load (BGE-M3/reranker rebuild here, AND the LLM this time)
  uses the same **timeout + retry + cache-lock cleanup** wrapper -- a
  unfinished load gets interrupted after a generous timeout and retried
  state, rather than hanging forever or corrupting itself on retry.
- **Incremental checkpointing**: saves progress after every single claim
  via true CSV append (not a full rewrite), and resumes automatically if
  interrupted -- safe against Colab disconnects.


In [1]:
# ===== Cell 1: Install — GPU cell =====
!pip -q install -U pip
!pip -q install transformers accelerate bitsandbytes sentence-transformers rank_bm25 faiss-cpu \
               pandas numpy tqdm pyarrow huggingface_hub sentencepiece protobuf
print('Dependencies installed.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.4 MB/s eta 0:00:00
Dependencies installed.


In [2]:
# ===== Cell 2: Mount Drive + checkpoint folder — CPU =====
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

RUN_NAME = 'averitec_20claim_v1'
PROJECT_DIR = Path('/content/drive/MyDrive/averitec_extended')
CHECKPOINT_DIR = PROJECT_DIR / RUN_NAME / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('Run:', RUN_NAME)
print('Checkpoint folder:', CHECKPOINT_DIR)


Mounted at /content/drive
Run: averitec_20claim_v1
Checkpoint folder: /content/drive/MyDrive/averitec_extended/averitec_20claim_v1/checkpoints


## Local Hugging Face cache + authentication

In [3]:
# ===== Cell 3: Local HF cache + authentication — CPU =====
import os, getpass

# Configure local cache before importing huggingface_hub or model libraries.
HF_CACHE_DIR = Path('/content/hf_cache')
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
print('HF model cache (local disk):', HF_CACHE_DIR)

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        print('Using token from Colab Secrets.')
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('No token found. Paste your HF token now (hidden), or press Enter to skip: ').strip()

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF token active.')
else:
    print('No HF token set -- recommended: set one via Colab Secrets (key icon, left sidebar).')


HF model cache (local disk): /content/hf_cache
Using token from Colab Secrets.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF token active.


In [4]:
# ===== Cell 4: Verify HF cache setup =====
assert os.environ['HF_HOME'] == str(HF_CACHE_DIR)
assert os.environ['HF_HUB_CACHE'] == str(HF_CACHE_DIR / 'hub')
print('HF cache configuration verified.')


HF cache configuration verified.


In [5]:
# ===== Cell 5: Imports, GPU check, settings =====
import re, json, time, math, signal
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss
from tqdm.auto import tqdm

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('\n*** No GPU detected. Runtime -> Change runtime type -> T4 GPU. ***')
    print('*** The LLM stage below will fail or be impractically slow on CPU. ***\n')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

RERANK_BATCH_SIZE, RERANK_MAX_LEN = 16, 512

LLM_MODEL_NAME     = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SUBCLAIMS      = 4
CONF_GATE_THRESH   = 0.85
MAX_EVIDENCE_CHARS = 500
MAX_CONTEXT_CHARS  = 3500
DECOMP_MAX_TOKENS  = 300
VERDICT_MAX_TOKENS = 130
ENABLE_PARAMETRIC_GATE = False  # False for evidence-grounded AVeriTeC evaluation
BASELINE_VERSION = 'v2'

LABELS = ['Supported', 'Refuted', 'Not Enough Evidence', 'Conflicting Evidence']
VALID_ATOMIC_LABELS = {'Supported', 'Refuted', 'Not Enough Evidence'}

TOKEN_RE = re.compile(r'[A-Za-z0-9]+')
def tokenize_bm25(text):
    return TOKEN_RE.findall(str(text).lower())

def minmax_normalize(values):
    values = np.asarray(values, dtype='float32')
    if values.size == 0:
        return values
    v_min, v_max = float(np.min(values)), float(np.max(values))
    if math.isclose(v_min, v_max):
        return np.zeros_like(values, dtype='float32')
    return (values - v_min) / (v_max - v_min)

def clean_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ''
    x = str(x)
    x = x.replace('\xa0', ' ')
    x = re.sub(r'\s+', ' ', x).strip()
    return x

def truncate_text(text, max_chars):
    text = clean_text(text)
    return text if len(text) <= max_chars else text[:max_chars].rsplit(' ', 1)[0] + ' ...'

def safe_confidence(value, default=0.5):
    try:
        return max(0.0, min(1.0, float(value)))
    except (TypeError, ValueError):
        return default

def ensure_list(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []

def normalize_label(text):
    if not isinstance(text, str):
        return 'Unknown'
    t = text.lower().strip()
    if any(k in t for k in ['not enough', 'insufficient', 'cannot determine', 'unknown', 'unverified']):
        return 'Not Enough Evidence'
    if any(k in t for k in ['conflicting', 'mixed', 'cherry']):
        return 'Conflicting Evidence'
    if any(k in t for k in ['refuted', 'false', 'incorrect', 'contradict']):
        return 'Refuted'
    if any(k in t for k in ['supported', 'true', 'correct', 'support']):
        return 'Supported'
    return 'Unknown'

def extract_json_list(text):
    text = re.sub(r'```(?:json)?', '', text).strip()
    try:
        s, e = text.find('['), text.rfind(']') + 1
        return json.loads(text[s:e]) if s != -1 and e != 0 else None
    except Exception:
        return None

def extract_json_obj(text):
    try:
        s, e = text.find('{'), text.rfind('}') + 1
        return json.loads(text[s:e]) if s != -1 and e != 0 else None
    except Exception:
        return None

print('Settings ready.')


CUDA available: True
GPU: Tesla T4
Settings ready.


## Robust model loading — timeout + retry + cache-lock cleanup
Same pattern that fixed Notebook 2's stalls, reused here for BOTH the
retrieval models and the LLM (which is an even bigger download).

In [6]:
# ===== Cell 6: Robust loading wrapper (shared by retrieval models + LLM) =====

class _LoadTimeout(Exception):
    pass

def _timeout_handler(signum, frame):
    raise _LoadTimeout()


def _clean_repo_cache_locks(repo_id):
    """Interrupting a download mid-transfer can leave huggingface_hub's
    cache with stale lock files. Remove locks before retrying while
    preserving .incomplete files so resumable progress is not lost."""
    cache_subdir = 'models--' + repo_id.replace('/', '--')
    repo_cache_path = HF_CACHE_DIR / 'hub' / cache_subdir
    if not repo_cache_path.exists():
        return
    removed = 0
    for pattern in ['*.lock']:
        for f in repo_cache_path.rglob(pattern):
            try:
                f.unlink()
                removed += 1
            except Exception:
                pass
    if removed:
        print(f'  Cleaned {removed} stale lock file(s) before retrying.')


def load_with_timeout_retry(load_fn, repo_id, max_attempts=4, attempt_timeout_sec=900):
    """Calls load_fn() (the plain automatic from_pretrained/SentenceTransformer/
    CrossEncoder call) with a supervisory timeout. If it remains unfinished,
    interrupt and retry after cleaning stale locks while preserving partial
    downloads."""
    for attempt in range(1, max_attempts + 1):
        print(f'Loading {repo_id}, attempt {attempt}/{max_attempts} '
              f'(will interrupt + retry if unfinished after {attempt_timeout_sec}s)...')
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(attempt_timeout_sec)
        try:
            result = load_fn()
            signal.alarm(0)
            return result
        except _LoadTimeout:
            print(f'  Load unfinished after {attempt_timeout_sec}s -- interrupting and retrying...')
        except Exception as ex:
            print(f'  Attempt {attempt} failed: {ex}')
        finally:
            signal.alarm(0)
        if attempt < max_attempts:
            _clean_repo_cache_locks(repo_id)
            time.sleep(5)
    raise RuntimeError(f'Could not load {repo_id} after {max_attempts} attempts. '
                       f'This suggests a sustained HF-side issue right now -- try again later.')

print('Robust loading wrapper ready.')


Robust loading wrapper ready.


In [7]:
# ===== Cell 7: Load Notebook 1+2 outputs + rebuild retrieval stack =====

required = ['chunks_df.parquet', 'test_df.parquet', 'chunk_embeddings.npy', 'faiss_index.bin', 'retrieval_config.json']
missing = [f for f in required if not (CHECKPOINT_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files: {missing}. Run Notebooks 1 and 2 first.')

chunks_df = pd.read_parquet(CHECKPOINT_DIR / 'chunks_df.parquet')
test_df   = pd.read_parquet(CHECKPOINT_DIR / 'test_df.parquet')
chunk_embeddings = np.load(CHECKPOINT_DIR / 'chunk_embeddings.npy')
faiss_index = faiss.read_index(str(CHECKPOINT_DIR / 'faiss_index.bin'))
with open(CHECKPOINT_DIR / 'retrieval_config.json') as f:
    retrieval_config = json.load(f)

BGE_MODEL_NAME = retrieval_config['BGE_MODEL_NAME']
RERANKER_MODEL_NAME = retrieval_config['RERANKER_MODEL_NAME']
BGE_QUERY_PREFIX = retrieval_config.get('BGE_QUERY_PREFIX', '')
BM25_TOP_N = int(retrieval_config['BM25_TOP_N'])
BGE_TOP_N = int(retrieval_config['BGE_TOP_N'])
FINAL_TOP_K = int(retrieval_config['FINAL_TOP_K'])
BM25_WEIGHT = float(retrieval_config['BM25_WEIGHT'])
BGE_WEIGHT = float(retrieval_config['BGE_WEIGHT'])
TOP_K_RERANKED = int(retrieval_config['TOP_K_RERANKED'])

if len(chunks_df) != len(chunk_embeddings):
    raise ValueError('Chunk/embedding count mismatch. Rerun Notebook 2 for this RUN_NAME.')
if faiss_index.ntotal != len(chunk_embeddings):
    raise ValueError('FAISS/embedding count mismatch. Rerun Notebook 2.')
if chunk_embeddings.ndim != 2 or faiss_index.d != chunk_embeddings.shape[1]:
    raise ValueError('FAISS/embedding dimension mismatch. Rerun Notebook 2.')
chunk_embeddings = chunk_embeddings.astype('float32', copy=False)

print(f'Loaded: {len(chunks_df)} chunks, {len(test_df)} test claims, '
      f'embeddings {chunk_embeddings.shape}, FAISS size {faiss_index.ntotal}')
print(f'Embedding model (from Notebook 2): {BGE_MODEL_NAME}')

print('Rebuilding BM25 (cheap, ~seconds)...')
tokenized_corpus = [tokenize_bm25(t) for t in chunks_df['retrieval_text'].tolist()]
bm25 = BM25Okapi(tokenized_corpus)
print('BM25 ready.')

bge_model = load_with_timeout_retry(
    lambda: SentenceTransformer(BGE_MODEL_NAME, device=DEVICE), BGE_MODEL_NAME
)
reranker = load_with_timeout_retry(
    lambda: CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE, max_length=RERANK_MAX_LEN), RERANKER_MODEL_NAME
)
print('Retrieval stack ready.')


def hybrid_retrieve(query_text, top_k=FINAL_TOP_K):
    tokenized_q = tokenize_bm25(query_text)
    bm25_scores_all = bm25.get_scores(tokenized_q)
    bm25_top_idx = np.argsort(bm25_scores_all)[::-1][:BM25_TOP_N]
    dense_query = BGE_QUERY_PREFIX + query_text
    q_emb = bge_model.encode([dense_query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    dense_scores, dense_idx = faiss_index.search(q_emb, BGE_TOP_N)
    dense_top_idx = dense_idx[0][dense_idx[0] >= 0].astype(int)
    union_idx = np.array(sorted(set(bm25_top_idx.tolist()) | set(dense_top_idx.tolist())), dtype=int)
    if len(union_idx) == 0:
        return []
    bm25_scores = bm25_scores_all[union_idx].astype('float32')
    bge_scores = np.dot(chunk_embeddings[union_idx], q_emb[0]).astype('float32')
    combined = BM25_WEIGHT * minmax_normalize(bm25_scores) + BGE_WEIGHT * minmax_normalize(bge_scores)
    order = np.argsort(combined)[::-1]
    selected = order[:min(top_k, len(order))]
    results = []
    for rank_pos, local_pos in enumerate(selected, start=1):
        chunk_idx = int(union_idx[local_pos])
        row = chunks_df.iloc[chunk_idx]
        results.append({'rank': rank_pos, 'chunk_id': row['chunk_id'], 'doc_id': row['doc_id'],
                        'chunk_text': row['chunk_text'], 'url': row.get('url', ''),
                        'combined_score': float(combined[local_pos])})
    return results


def rerank_chunks(query_text, chunks, top_k=TOP_K_RERANKED):
    if not chunks:
        return []
    pairs = [(query_text, c['chunk_text']) for c in chunks]
    scores = reranker.predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False, convert_to_numpy=True)
    order = np.argsort(scores)[::-1]
    reranked = []
    for new_rank, idx in enumerate(order[:top_k], start=1):
        c = dict(chunks[idx]); c['rerank_rank'] = new_rank; c['rerank_score'] = float(scores[idx])
        reranked.append(c)
    return reranked


Loaded: 634 chunks, 20 test claims, embeddings (634, 1024), FAISS size 634
Embedding model (from Notebook 2): BAAI/bge-large-en-v1.5
Rebuilding BM25 (cheap, ~seconds)...
BM25 ready.
Loading BAAI/bge-large-en-v1.5, attempt 1/4 (will interrupt + retry if unfinished after 900s)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retrieval stack ready.


## Load LLM (4-bit) — GPU REQUIRED, uses the same robust loading wrapper

In [8]:
# ===== Cell 8: Load LLM (4-bit) =====

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

llm_tokenizer = load_with_timeout_retry(
    lambda: AutoTokenizer.from_pretrained(LLM_MODEL_NAME, token=HF_TOKEN), LLM_MODEL_NAME
)
if llm_tokenizer.pad_token_id is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

llm_model = load_with_timeout_retry(
    lambda: AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME, token=HF_TOKEN, device_map='auto',
        torch_dtype=torch.float16, quantization_config=quantization_config,
    ),
    LLM_MODEL_NAME, attempt_timeout_sec=1200,  # allow a healthy first 3B-model download to finish
)
llm_model.eval()
print('LLM ready.')


def llm_generate(prompt, max_new_tokens=200, system_prompt=None):
    sys_msg = system_prompt or 'Reply only with the requested format. No extra explanation.'
    messages = [{'role': 'system', 'content': sys_msg}, {'role': 'user', 'content': prompt}]
    if getattr(llm_tokenizer, 'chat_template', None):
        text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f'[INST] <<SYS>>\n{sys_msg}\n<</SYS>>\n\n{prompt}\n[/INST]'
    inputs = llm_tokenizer(text, return_tensors='pt', truncation=True, max_length=4096).to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=llm_tokenizer.eos_token_id)
    new_toks = out[0][inputs['input_ids'].shape[-1]:]
    return llm_tokenizer.decode(new_toks, skip_special_tokens=True).strip()


Loading Qwen/Qwen2.5-3B-Instruct, attempt 1/4 (will interrupt + retry if unfinished after 900s)...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM ready.


## Decomposition, confidence gate, verdict, aggregation

In [9]:
# ===== Cell 9: Decomposition + reflection =====

DECOMPOSE_PROMPT = """Break the following claim into independent, verifiable sub-claims.

Rules:
- Each sub-claim must contain ONLY facts that appear word-for-word in the original claim
- Each sub-claim must be self-contained (replace pronouns with actual names/entities)
- Do NOT add, infer, or invent ANY information not explicitly stated in the original claim
- Maximum {max_subclaims} sub-claims
- If the claim is already short and atomic, return it as a single-item list

Return ONLY a JSON list of strings, no explanation, no markdown, no code fences:
[\"sub-claim 1\", \"sub-claim 2\"]

Claim: {claim}"""

REFLECT_PROMPT = """Check these sub-claims against the original claim for errors.

Error types to fix:
1. OMISSION: key fact from original is missing
2. AMBIGUITY: uses unclear pronouns or incomplete names
3. OVER_DECOMP: trivially obvious or redundant sub-claim
4. ALTERATION: contains ANY information not in the original -- DELETE these

Original claim: {claim}
Sub-claims: {subclaims}

Remove any sub-claim that adds information not in the original.
Return ONLY a corrected JSON list, no explanation:
[\"sub-claim 1\", ...]"""


def decompose_claim(claim):
    prompt = DECOMPOSE_PROMPT.format(claim=claim, max_subclaims=MAX_SUBCLAIMS)
    raw = llm_generate(prompt, max_new_tokens=DECOMP_MAX_TOKENS,
                        system_prompt='You are a precise fact-checking assistant. Follow instructions exactly. Never add information not present in the original claim.')
    result = extract_json_list(raw)
    if not result or not isinstance(result, list):
        return [claim]
    result = [clean_text(s) for s in result if clean_text(s)]
    result = [s for s in result if len(s.split()) <= len(claim.split()) * 2]
    return result[:MAX_SUBCLAIMS] or [claim]


def reflect_and_refine(claim, subclaims):
    prompt = REFLECT_PROMPT.format(claim=claim, subclaims=json.dumps(subclaims, ensure_ascii=False))
    raw = llm_generate(prompt, max_new_tokens=DECOMP_MAX_TOKENS)
    refined = extract_json_list(raw)
    if not refined or not isinstance(refined, list):
        return subclaims
    refined = [clean_text(s) for s in refined if clean_text(s)]
    return refined[:MAX_SUBCLAIMS] or subclaims

print('Decomposition functions ready.')


Decomposition functions ready.


In [10]:
# ===== Cell 10: Confidence gate + baseline verdict + deterministic aggregation =====

CONFIDENCE_PROMPT = """Rate your confidence in verifying this sub-claim using only your training knowledge.

Sub-claim: {subclaim}

Reply ONLY with JSON, no explanation:
{{\"confidence\": 0.0_to_1.0, \"preliminary_label\": \"Supported\" or \"Refuted\" or \"Not Enough Evidence\"}}"""

ATOMIC_VERDICT_PROMPT = """Verify the following sub-claim using ONLY the evidence provided.

Sub-claim: {subclaim}

Evidence:
{evidence}

Rules:
- Use only the evidence above
- Do not use outside knowledge
- Choose exactly one label: Supported, Refuted, Not Enough Evidence
- Cite only the numbered evidence items actually used

Reply ONLY with JSON:
{{\"label\": \"Supported\" or \"Refuted\" or \"Not Enough Evidence\", \"confidence\": 0.0_to_1.0, \"key_evidence\": \"one phrase from evidence\", \"evidence_ids\": [1, 3]}}"""


def build_evidence_context(chunks):
    parts, total = [], 0
    for c in chunks:
        text = truncate_text(c.get('chunk_text', ''), MAX_EVIDENCE_CHARS)
        url = c.get('url', '')
        part = f"[{c.get('rerank_rank', c.get('rank', 1))}] URL: {url}\n{text}"
        if total + len(part) > MAX_CONTEXT_CHARS:
            break
        parts.append(part)
        total += len(part)
    return '\n\n'.join(parts)


def check_confidence(subclaim):
    prompt = CONFIDENCE_PROMPT.format(subclaim=subclaim)
    raw = llm_generate(prompt, max_new_tokens=80)
    parsed = extract_json_obj(raw)
    if not parsed:
        return 0.0, None
    label = normalize_label(str(parsed.get('preliminary_label', '')))
    return safe_confidence(parsed.get('confidence', 0.0), 0.0), label


def select_cited_chunks(chunks, evidence_ids):
    wanted = set()
    for value in ensure_list(evidence_ids):
        try:
            wanted.add(int(value))
        except (TypeError, ValueError):
            continue
    return [c for c in chunks if int(c.get('rerank_rank', c.get('rank', -1))) in wanted]


def atomic_verdict(subclaim, chunks):
    evidence_text = build_evidence_context(chunks)
    if not evidence_text.strip():
        return {'label': 'Not Enough Evidence', 'confidence': 0.1, 'key_evidence': 'no evidence',
                'used_retrieval': True, 'cited_chunks': []}
    prompt = ATOMIC_VERDICT_PROMPT.format(subclaim=subclaim, evidence=evidence_text)
    raw = llm_generate(prompt, max_new_tokens=VERDICT_MAX_TOKENS)
    parsed = extract_json_obj(raw)
    if parsed:
        label = normalize_label(str(parsed.get('label', 'Not Enough Evidence')))
        if label not in VALID_ATOMIC_LABELS:
            label = 'Not Enough Evidence'
        cited = select_cited_chunks(chunks, parsed.get('evidence_ids', []))
        return {'label': label,
                'confidence': safe_confidence(parsed.get('confidence', 0.5), 0.5),
                'key_evidence': clean_text(str(parsed.get('key_evidence', ''))),
                'used_retrieval': True, 'cited_chunks': cited}
    fallback_label = normalize_label(raw)
    if fallback_label not in VALID_ATOMIC_LABELS:
        fallback_label = 'Not Enough Evidence'
    return {'label': fallback_label, 'confidence': 0.4, 'key_evidence': 'json parse failed',
            'used_retrieval': True, 'cited_chunks': []}


def aggregate_4class(atomic_results):
    if not atomic_results:
        return 'Not Enough Evidence', 0.0, {}
    labels = [r.get('label') if r.get('label') in VALID_ATOMIC_LABELS else 'Not Enough Evidence'
              for r in atomic_results]
    confs = [safe_confidence(r.get('confidence', 0.5), 0.5) for r in atomic_results]
    total = sum(confs) if sum(confs) > 0 else 1.0
    w = {'Supported': sum(c for l, c in zip(labels, confs) if l == 'Supported'),
         'Refuted': sum(c for l, c in zip(labels, confs) if l == 'Refuted'),
         'Not Enough Evidence': sum(c for l, c in zip(labels, confs) if l == 'Not Enough Evidence')}
    ratios = {k: v / total for k, v in w.items()}
    if ratios['Supported'] > 0.20 and ratios['Refuted'] > 0.20:
        return 'Conflicting Evidence', round(min(ratios['Supported'], ratios['Refuted']), 3), ratios
    winner = max(ratios, key=ratios.get)
    return winner, round(ratios[winner], 3), ratios

print('Baseline verdict + aggregation functions ready.')


Baseline verdict + aggregation functions ready.


## MAIN LOOP — GPU REQUIRED, with incremental checkpoint + resume (true append)
Safe to interrupt and re-run this cell -- already-completed claims are
skipped, and progress is saved after every single claim.

In [11]:
# ===== Cell 11: Main baseline loop =====

PARTIAL_PATH = CHECKPOINT_DIR / f'predictions_baseline_{BASELINE_VERSION}_partial.csv'
FINAL_PATH   = CHECKPOINT_DIR / 'predictions_baseline.parquet'
CONFIG_PATH  = CHECKPOINT_DIR / 'baseline_config.json'
PRINT_EVERY  = 5

if PARTIAL_PATH.exists():
    done_df = pd.read_csv(PARTIAL_PATH)
    done_ids = set(done_df['claim_id'].astype(str))
    results = done_df.to_dict('records')
    print(f'Resuming: {len(done_ids)} claims already done, skipping them.')
else:
    done_ids = set()
    results = []

def append_row_to_csv(row_dict, path, header_written):
    """True append -- writes ONE row, not the whole growing file."""
    row_df = pd.DataFrame([row_dict])
    row_df.to_csv(path, mode='a', header=not header_written, index=False)
    return True

_header_already_written = PARTIAL_PATH.exists() and PARTIAL_PATH.stat().st_size > 0

start_time = time.time()
remaining = test_df[~test_df['claim_id'].astype(str).isin(done_ids)]
print(f'{len(remaining)} claims remaining out of {len(test_df)} total.')

for idx, row in tqdm(remaining.iterrows(), total=len(remaining), desc='Baseline pipeline'):
    claim_id, claim, gold_label = str(row['claim_id']), clean_text(row['claim']), row['label']
    gold_qa = ensure_list(row.get('gold_qa_pairs', []))
    t0 = time.time()

    try:
        subclaims = decompose_claim(claim)
        subclaims = reflect_and_refine(claim, subclaims)
    except Exception as ex:
        subclaims = [claim]
        print(f'  [WARN] decompose {claim_id}: {ex}')

    atomic_results, retrieval_skipped = [], 0
    for sc in subclaims:
        try:
            conf, prelim = check_confidence(sc)
        except Exception:
            conf, prelim = 0.0, None
        if ENABLE_PARAMETRIC_GATE and conf >= CONF_GATE_THRESH and prelim in VALID_ATOMIC_LABELS:
            atomic_results.append({'label': prelim, 'confidence': conf, 'key_evidence': 'parametric',
                                    'used_retrieval': False, 'cited_chunks': []})
            retrieval_skipped += 1
        else:
            try:
                candidates = hybrid_retrieve(sc, top_k=FINAL_TOP_K)
                reranked = rerank_chunks(sc, candidates, top_k=TOP_K_RERANKED)
                atomic_results.append(atomic_verdict(sc, reranked))
            except Exception as ex:
                atomic_results.append({'label': 'Not Enough Evidence', 'confidence': 0.1,
                                       'key_evidence': f'error: {ex}', 'used_retrieval': True, 'cited_chunks': []})

    final_label, agg_conf, ratios = aggregate_4class(atomic_results)
    elapsed = round(time.time() - t0, 1)
    correct = 'CORRECT' if final_label == gold_label else 'WRONG'

    pred_qa_pairs = [{'question': sc, 'answer': ar.get('key_evidence', '')} for sc, ar in zip(subclaims, atomic_results)]
    justification = ' '.join(ar.get('key_evidence', '') for ar in atomic_results if ar.get('key_evidence'))

    new_row = {
        'claim_id': claim_id, 'claim': claim, 'gold_label': gold_label,
        'predicted_label': final_label, 'agg_confidence': agg_conf, 'agg_method': 'deterministic',
        'n_subclaims': len(subclaims), 'subclaims_json': json.dumps(subclaims, ensure_ascii=False),
        'atomic_results_json': json.dumps(atomic_results, ensure_ascii=False, default=str),
        'weight_ratios_json': json.dumps(ratios, ensure_ascii=False),
        'retrieval_skipped': retrieval_skipped,
        'pred_qa_pairs': json.dumps(pred_qa_pairs, ensure_ascii=False),
        'gold_qa_pairs': json.dumps(gold_qa, ensure_ascii=False, default=str),
        'justification': justification, 'elapsed_seconds': elapsed,
    }
    results.append(new_row)
    _header_already_written = append_row_to_csv(new_row, PARTIAL_PATH, _header_already_written)

    if len(results) % PRINT_EVERY == 0 or len(results) == len(test_df):
        print(f'{correct:8s} [{len(results):03d}/{len(test_df)}] {claim_id} '
              f'Gold={gold_label:25s} Pred={final_label:25s} {len(subclaims)} sub-claims {elapsed}s')

predictions_df = pd.DataFrame(results)
predictions_df.to_parquet(FINAL_PATH, index=False)
baseline_config = {
    'run_name': RUN_NAME, 'baseline_version': BASELINE_VERSION,
    'llm_model': LLM_MODEL_NAME, 'enable_parametric_gate': ENABLE_PARAMETRIC_GATE,
    'confidence_gate_threshold': CONF_GATE_THRESH, 'max_subclaims': MAX_SUBCLAIMS,
    'retrieval_config': retrieval_config,
}
with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(baseline_config, f, indent=2, ensure_ascii=False)
total_min = round((time.time() - start_time) / 60, 1)
print(f'\nBaseline pipeline finished: {len(predictions_df)} claims in {total_min} min (this run).')
print(f'Saved: {FINAL_PATH}')
print(f'Saved: {CONFIG_PATH}')
print()
print('DONE. Open Notebook 4 (Retrieval Novelties) next.')


20 claims remaining out of 20 total.


Baseline pipeline:   0%|          | 0/20 [00:00<?, ?it/s]

CORRECT  [005/20] 98 Gold=Supported                 Pred=Supported                 1 sub-claims 10.4s
WRONG    [010/20] 282 Gold=Not Enough Evidence       Pred=Supported                 1 sub-claims 7.3s
WRONG    [015/20] 96 Gold=Refuted                   Pred=Supported                 1 sub-claims 10.3s
WRONG    [020/20] 193 Gold=Refuted                   Pred=Supported                 1 sub-claims 6.1s

Baseline pipeline finished: 20 claims in 4.2 min (this run).
Saved: /content/drive/MyDrive/averitec_extended/averitec_20claim_v1/checkpoints/predictions_baseline.parquet
Saved: /content/drive/MyDrive/averitec_extended/averitec_20claim_v1/checkpoints/baseline_config.json

DONE. Open Notebook 4 (Retrieval Novelties) next.
